In [0]:
# CELL 1 — Install dependencies
%pip install confluent-kafka jsonschema requests websockets

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# CELL 2 — Configuration
import os

# Option A : Databricks Secrets (production)
def get_secret(key, fallback=""):
    try:
        return dbutils.secrets.get(scope="crypto-intel", key=key)
    except:
        return fallback

# Option B : hardcode temporaire pour tester (à supprimer après)
CONFLUENT_BOOTSTRAP_SERVERS = get_secret("CONFLUENT_BOOTSTRAP_SERVERS", "pkc-921jm.us-east-2.aws.confluent.cloud:9092")
CONFLUENT_API_KEY            = get_secret("CONFLUENT_API_KEY",            "GLLKB5UTLMEPUJHW")
CONFLUENT_API_SECRET         = get_secret("CONFLUENT_API_SECRET",         "cfltI+pm4Cwo5YSUgSp5kOmT/LJII/hlASGiIDOENx8iF3+QCsax3TD550sbE79A")
SUPABASE_JDBC_URL            = get_secret("SUPABASE_JDBC_URL",            "jdbc:postgresql://db.apxdzojufohvsdgjobds.supabase.co:5432/postgres")
SUPABASE_USER                = get_secret("SUPABASE_USER",                "postgres")
SUPABASE_PASSWORD            = get_secret("SUPABASE_PASSWORD",            "L3woy4Qyt8o1VPe3")

CHECKPOINT_PATH = "/tmp/checkpoints/crypto-intel"
DATA_LAKE_PATH  = "/tmp/data/crypto-intel"

print(" Config chargée")
print(f"   Bootstrap : {CONFLUENT_BOOTSTRAP_SERVERS[:30]}...")
print(f"   Data Lake : {DATA_LAKE_PATH}")

 Config chargée
   Bootstrap : pkc-921jm.us-east-2.aws.conflu...
   Data Lake : /tmp/data/crypto-intel


In [0]:
# CELL 3 — Schemas
from pyspark.sql.types import *

TRADE_SCHEMA = StructType([
    StructField("event_type",     StringType(),  True),
    StructField("event_time",     LongType(),    True),
    StructField("symbol",         StringType(),  False),
    StructField("trade_id",       LongType(),    True),
    StructField("price",          StringType(),  False),
    StructField("quantity",       StringType(),  False),
    StructField("trade_time",     LongType(),    False),
    StructField("is_buyer_maker", BooleanType(), True),
    StructField("ingested_at",    StringType(),  True),
])

NEWS_SCHEMA = StructType([
    StructField("article_id",   StringType(), False),
    StructField("title",        StringType(), True),
    StructField("description",  StringType(), True),
    StructField("url",          StringType(), True),
    StructField("published_at", StringType(), False),
    StructField("keywords",     ArrayType(StringType()), True),
    StructField("ingested_at",  StringType(), True),
])

ENRICHED_SCHEMA = StructType([
    StructField("symbol",                StringType(),    False),
    StructField("window_start",          TimestampType(), True),
    StructField("window_end",            TimestampType(), True),
    StructField("open_price",            DoubleType(),    True),
    StructField("close_price",           DoubleType(),    True),
    StructField("high_price",            DoubleType(),    True),
    StructField("low_price",             DoubleType(),    True),
    StructField("avg_price",             DoubleType(),    True),
    StructField("total_volume",          DoubleType(),    True),
    StructField("trade_count",           IntegerType(),   True),
    StructField("price_change_pct",      DoubleType(),    True),
    StructField("volatility_score",      DoubleType(),    True),
    StructField("is_spike",              BooleanType(),   True),
    StructField("related_article_id",    StringType(),    True),
    StructField("related_article_title", StringType(),    True),
    StructField("related_article_url",   StringType(),    True),
    StructField("fred_interest_rate",    DoubleType(),    True),
    StructField("fred_inflation_rate",   DoubleType(),    True),
    StructField("enriched_at",           TimestampType(), True),
])

print("✅ Schemas définis")

✅ Schemas définis


In [0]:
# CELL 4 — Kafka helpers
from pyspark.sql import functions as F

JAAS = (
    f'org.apache.kafka.common.security.plain.PlainLoginModule required '
    f'username="{CONFLUENT_API_KEY}" '
    f'password="{CONFLUENT_API_SECRET}";'
)

def kafka_read_options(topic):
    return {
        "kafka.bootstrap.servers": CONFLUENT_BOOTSTRAP_SERVERS,
        "kafka.security.protocol": "SASL_SSL",
        "kafka.sasl.mechanism":    "PLAIN",
        "kafka.sasl.jaas.config":  JAAS,
        "subscribe":               topic,
        "startingOffsets":         "latest",
        "failOnDataLoss":          "false",
    }

KAFKA_WRITE_CONFIG = {
    "kafka.bootstrap.servers": CONFLUENT_BOOTSTRAP_SERVERS,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism":    "PLAIN",
    "kafka.sasl.jaas.config":  JAAS,
}

print("✅ Kafka config prête")

✅ Kafka config prête


In [0]:
# CELL 5 — Read streams (trades + news)

# Stream trades avec watermark
trades_raw = (
    spark.readStream.format("kafka")
    .options(**kafka_read_options("trades_topic"))
    .load()
    .select(F.from_json(F.col("value").cast("string"), TRADE_SCHEMA).alias("d"))
    .select("d.*")
    .withColumn("price_f",  F.col("price").cast(DoubleType()))
    .withColumn("qty_f",    F.col("quantity").cast(DoubleType()))
    .withColumn("trade_ts", (F.col("trade_time") / 1000).cast(TimestampType()))
    .withWatermark("trade_ts", "5 minutes")   
)

# Stream news avec watermark
news_raw = (
    spark.readStream.format("kafka")
    .options(**kafka_read_options("news_topic"))
    .load()
    .select(F.from_json(F.col("value").cast("string"), NEWS_SCHEMA).alias("d"))
    .select("d.*")
    .withColumn("news_ts", F.to_timestamp("published_at"))
    .withWatermark("news_ts", "10 minutes")   
)

print(" Streams trades + news définis (pas encore démarrés)")

 Streams trades + news définis (pas encore démarrés)


In [0]:
# CELL 6 — Windowing 1min + spike detection

spikes = (
    trades_raw
    .groupBy("symbol", F.window("trade_ts", "1 minute"))
    .agg(
        F.first("price_f").alias("open_price"),
        F.last("price_f").alias("close_price"),
        F.min("price_f").alias("low_price"),
        F.max("price_f").alias("high_price"),
        F.avg("price_f").alias("avg_price"),
        F.sum("qty_f").alias("total_volume"),
        F.count("*").alias("trade_count"),
        F.stddev("price_f").alias("price_stddev"),
    )
    .withColumn(
        "price_change_pct",
        ((F.col("close_price") - F.col("open_price")) / F.col("open_price")) * 100
    )
    .withColumn("is_spike", F.abs(F.col("price_change_pct")) >= 5.0)
    .withColumn(
        "volatility_score",
        F.least(
            F.coalesce(F.col("price_stddev") / F.col("avg_price"), F.lit(0.0)),
            F.lit(1.0)
        )
    )
    .withColumn("window_start", F.col("window.start"))
    .withColumn("window_end",   F.col("window.end"))
    .drop("window", "price_stddev")
)

print("✅ Window 1min + spike detection définis")

✅ Window 1min + spike detection définis


In [0]:
# CELL 7 — Stream-to-Stream Join news + price spike ±5 min

joined = (
    spikes.alias("s")
    .join(
        news_raw.alias("n"),
        F.expr("""
            n.news_ts >= s.window_start - INTERVAL 5 MINUTES AND
            n.news_ts <= s.window_end   + INTERVAL 5 MINUTES
        """),
        how="left"
    )
    .select(
        "s.symbol",
        "s.window_start",
        "s.window_end",
        "s.open_price",
        "s.close_price",
        "s.high_price",
        "s.low_price",
        "s.avg_price",
        "s.total_volume",
        "s.trade_count",
        "s.price_change_pct",
        "s.volatility_score",
        "s.is_spike",
        F.col("n.article_id").alias("related_article_id"),
        F.col("n.title").alias("related_article_title"),
        F.col("n.url").alias("related_article_url"),
        F.lit(None).cast(DoubleType()).alias("fred_interest_rate"),
        F.lit(None).cast(DoubleType()).alias("fred_inflation_rate"),
        F.current_timestamp().alias("enriched_at"),
    )
)

print("✅ Stream-to-Stream Join défini")

✅ Stream-to-Stream Join défini


In [0]:
# CELL 8 — Multi-Sink Fan-Out (foreachBatch)
import json, uuid
from datetime import datetime, timezone

# ── Sink 1 : Data Lake ─────────────────────────────────────────────
def sink_data_lake(batch, batch_id):
    if batch.isEmpty(): return
    (
        batch
        .withColumn("date", F.to_date("window_start"))
        .write.mode("append")
        .partitionBy("symbol", "date")
        .parquet(f"{DATA_LAKE_PATH}/enriched/")
    )
    print(f"  [DataLake]  batch={batch_id}")

# ── Sink 2 : Supabase ──────────────────────────────────────────────
JDBC_PROPS = {
    "user": SUPABASE_USER, "password": SUPABASE_PASSWORD,
    "driver": "org.postgresql.Driver", "ssl": "true", "sslmode": "require",
}
COLS = ["symbol","window_start","window_end","open_price","close_price",
        "avg_price","total_volume","trade_count","price_change_pct",
        "volatility_score","is_spike","related_article_url","enriched_at"]

def sink_supabase(batch, batch_id):
    if batch.isEmpty(): return
    existing = [f.name for f in batch.schema]
    (
        batch.select(*[c for c in COLS if c in existing])
        .write.jdbc(SUPABASE_JDBC_URL, "enriched_market_data", "append", JDBC_PROPS)
    )
    print(f"  [Supabase]  batch={batch_id}")

# ── Sink 3 : alerts_topic ──────────────────────────────────────────
def sink_alerts(batch, batch_id):
    rows = batch.filter(
        (F.col("is_spike") == True) | (F.col("volatility_score") > 0.8)
    ).collect()
    if not rows: return

    from confluent_kafka import Producer
    p = Producer({
        "bootstrap.servers": CONFLUENT_BOOTSTRAP_SERVERS,
        "security.protocol": "SASL_SSL",
        "sasl.mechanisms":   "PLAIN",
        "sasl.username":     CONFLUENT_API_KEY,
        "sasl.password":     CONFLUENT_API_SECRET,
    })
    now = datetime.now(timezone.utc).isoformat()
    for row in rows:
        alert = {
            "alert_id":            str(uuid.uuid4()),
            "alert_type":          "PRICE_SPIKE" if row.is_spike else "HIGH_VOLATILITY",
            "severity":            "HIGH" if abs(row.price_change_pct or 0) >= 10 else "MEDIUM",
            "symbol":              row.symbol,
            "message":             f"{row.symbol} spike {row.price_change_pct:.2f}%" if row.is_spike else f"{row.symbol} volatility={row.volatility_score:.3f}",
            "price_at_trigger":    row.close_price,
            "price_change_pct":    row.price_change_pct,
            "volatility_score":    row.volatility_score,
            "related_article_url": row.related_article_url,
            "window_start":        str(row.window_start),
            "window_end":          str(row.window_end),
            "triggered_at":        now,
        }
        p.produce("alerts_topic", key=row.symbol.encode(), value=json.dumps(alert).encode())
    p.flush(10)
    print(f"  [alerts_topic]  batch={batch_id} → {len(rows)} alerts")

# ── foreachBatch orchestrateur ─────────────────────────────────────
def process_batch(batch, batch_id):
    print(f"\n Batch {batch_id} — count={batch.count()}")
    batch.cache()
    for name, fn in [("DataLake", sink_data_lake), ("Supabase", sink_supabase), ("Kafka", sink_alerts)]:
        try:
            fn(batch, batch_id)
        except Exception as e:
            print(f"   [{name}] {e}")
    batch.unpersist()

print("foreachBatch défini")

✅ foreachBatch défini


In [0]:
# CELL 9 — START le streaming query

query = (
    joined
    .writeStream
    .foreachBatch(process_batch)
    .option("checkpointLocation", f"{CHECKPOINT_PATH}/fanout")
    .trigger(availableNow=True)
    .start()
)

print(" Pipeline démarré !")
print(f"   Status : {query.status}")

---------------------------------------------------------------------------
UnsupportedOperationException             Traceback (most recent call last)
File <command-8924240933782183>, line 9
      1 # CELL 9 — START le streaming query
      3 query = (
      4     joined
      5     .writeStream
      6     .foreachBatch(process_batch)
      7     .option("checkpointLocation", f"{CHECKPOINT_PATH}/fanout")
      8     .trigger(availableNow=True)
----> 9     .start()
     10 )
     12 print(" Pipeline démarré !")
     13 print(f"   Status : {query.status}")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/streaming/readwriter.py:713, in DataStreamWriter.start(self, path, format, outputMode, partitionBy, queryName, **options)
    704 def start(
    705     self,
    706     path: Optional[str] = None,
   (...)
    711     **options: "OptionalPrimitiveType",
    712 ) -> "StreamingQuery":
--> 713     return self._start_internal(
    714         path=path,
    715 

In [0]:
# CELL 11 — Vérifier que les données arrivent

# Data Lake
print("=== DATA LAKE ===")
try:
    df = spark.read.parquet(f"{DATA_LAKE_PATH}/enriched/")
    print(f"Rows : {df.count()}")
    df.show(3, truncate=False)
except Exception as e:
    print(f"Pas encore de données : {e}")

# Supabase (via JDBC)
print("\n=== SUPABASE ===")
try:
    df_supa = spark.read.jdbc(SUPABASE_JDBC_URL, "enriched_market_data", properties=JDBC_PROPS)
    print(f"Rows : {df_supa.count()}")
    df_supa.show(3)
except Exception as e:
    print(f"Erreur Supabase : {e}")

=== DATA LAKE ===
Pas encore de données : [DBFS_DISABLED] Public DBFS root is disabled. Access is denied on path: /tmp/data/crypto-intel/enriched SQLSTATE: 56038

JVM stacktrace:
com.databricks.backend.daemon.data.client.DbfsUnsupportedOperationSparkException
	at com.databricks.backend.daemon.data.client.DbfsExceptionMapperImpl.withExceptionWrapping(DbfsSparkException.scala:42)
	at com.databricks.backend.daemon.data.client.DBFSV2.getFileStatus(DatabricksFileSystemV2.scala:750)
	at com.databricks.backend.daemon.data.client.DatabricksFileSystem.getFileStatus(DatabricksFileSystem.scala:212)
	at com.databricks.sql.io.LokiFileSystem.$anonfun$getFileStatusNoCache$4(LokiFileSystem.scala:347)
	at com.databricks.sql.io.LokiFileSystem.tryWithNativeIO(LokiFileSystem.scala:319)
	at com.databricks.sql.io.LokiFileSystem.$anonfun$getFileStatusNoCache$1(LokiFileSystem.scala:347)
	at com.databricks.sql.io.LokiFileSystem$.withWrappedExceptions(LokiFileSystem.scala:165)
	at com.databricks.sql.io.LokiFile